# Day 23 — Pomo Capstone — Design & Schema

> ⚠️ **Why this matters.** Time to apply EVERY skill from Phase 1 on a new project — `pomo`, a Pomodoro tracker CLI. Different domain than english-helper, same tools and patterns. By proving you can do it twice, you prove the skills are yours, not the project's.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/23-pomo-capstone-design.ipynb)

## What you'll build

A CLI tool `pomo` that:

- Starts a Pomodoro session (25 min default), with an optional tag
- Stops the active session and records it
- Lists sessions in a date range
- Reports time by tag

Persisted to a SQLite database (introduces SQLite — easy with stdlib).

## What you'll do today (Day 23)

- [ ] Write a DESIGN.md for pomo before code
- [ ] Set up the project (uv init, package structure, CI from Day 20)
- [ ] Define the SQLite schema
- [ ] Implement and test the Storage class

## Tomorrow (Day 24)

- [ ] CLI with argparse
- [ ] Tests with mocks
- [ ] Package + install

## DESIGN.md (write before code)

Spend 30 minutes writing this. Real, in your repo.

### Problem
I want to track how I spend my study time. Currently I forget which subject I worked on. A Pomodoro tracker that knows the tag (e.g. "phase-1", "english-helper", "workout") would let me look back at where the time went.

### Users
Just me. One machine. No need for cloud sync.

### MVP scope
- `pomo start [--duration MIN] [--tag TAG]`
- `pomo stop`
- `pomo list [--since RANGE]`
- `pomo report [--since RANGE]`

### Out of scope
- Timers / notifications during a session
- Multi-user
- Sync between machines

### Data model
```sql
CREATE TABLE sessions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    started_at TEXT NOT NULL,   -- ISO 8601
    ended_at TEXT,              -- NULL = still running
    duration_min INTEGER NOT NULL,
    tag TEXT
);
```

### Constraints
- At most one running session at a time (no `ended_at`)
- `duration_min` must be > 0
- Storage at `~/.pomo/pomo.db`

## Why SQLite?

Built into Python's stdlib (`import sqlite3`). One file. ACID transactions. Queryable with SQL. Perfect for a single-user CLI tool.

**You'll learn SQL properly in Phase 2.** Today's SQL is light — just CREATE, INSERT, SELECT, UPDATE.

In [ ]:
# storage.py (sketch)
import sqlite3
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime

@dataclass
class Session:
    id: int | None
    started_at: datetime
    ended_at: datetime | None
    duration_min: int
    tag: str | None = None

class Storage:
    def __init__(self, path: Path):
        self.path = path
        path.parent.mkdir(exist_ok=True)
        self.conn = sqlite3.connect(path)
        self._init_schema()

    def _init_schema(self):
        self.conn.executescript('''
            CREATE TABLE IF NOT EXISTS sessions (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                started_at TEXT NOT NULL,
                ended_at TEXT,
                duration_min INTEGER NOT NULL,
                tag TEXT
            )
        ''')
        self.conn.commit()

    def start(self, duration_min: int, tag: str | None) -> Session:
        # Check no active session
        active = self.conn.execute(
            'SELECT id FROM sessions WHERE ended_at IS NULL'
        ).fetchone()
        if active:
            raise ValueError('A session is already running')
        started = datetime.now()
        cur = self.conn.execute(
            'INSERT INTO sessions (started_at, duration_min, tag) VALUES (?, ?, ?)',
            (started.isoformat(), duration_min, tag),
        )
        self.conn.commit()
        return Session(id=cur.lastrowid, started_at=started, ended_at=None, duration_min=duration_min, tag=tag)

## End-of-day Day 23 task

1. `uv init pomo`. Switch to `src/` layout.
2. Write `DESIGN.md` (the section above is a template — make it yours).
3. Implement `Session` dataclass + `Storage` class with `start`, `stop`, `list`, `report` methods.
4. Write tests in `tests/test_storage.py` — use `tmp_path` for the SQLite file.
5. Verify with manual demo:
   ```python
   s = Storage(Path('/tmp/test.db'))
   s.start(25, 'phase-1')
   import time; time.sleep(2)
   s.stop()
   print(s.list())
   ```
6. CI runs tests green.

**Tomorrow (Day 24):** the CLI layer and packaging.

## Connect to the project

> 🎯 **Tomorrow:** wire the CLI on top of today's Storage.

**Quiz:** [23-pomo-capstone-design-quiz.ipynb](23-pomo-capstone-design-quiz.ipynb)